# Notebook 09 — Data Cleaning for Biological Data

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 09 of 20  
**Prerequisites:** Notebook 08  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Real biological datasets are dirty. Gene IDs collide across databases. Samples fail
QC. Library sizes vary by orders of magnitude. Batch effects dominate the first
principal component. None of these problems show up as Python errors — they produce
silent wrong answers. Data cleaning is where most analysis time goes in practice.

---
## Step 2 — Intuition

Think of data cleaning as boundary enforcement: the raw data is untrusted input, and
the clean DataFrame is the validated, typed, documented output. Every transformation
should be logged, reversible (keep the raw file), and tested.

The six biological data cleaning problems: missing values, duplicated gene IDs,
low-count genes, outlier samples, gene ID format mismatches, batch effects.

---
## Step 3 — Biological Background

**Why low-count genes are filtered:**  
In RNA-seq, a gene with counts 0, 0, 1 across 3 samples has no statistical power for
differential expression — any p-value computed for it is unreliable. The conventional
filter: keep genes where at least $k$ samples have counts ≥ $c$ (typically $c=10$, $k$
= smallest group size).

**Gene ID formats:**  
- Ensembl: `ENSG00000012048` (BRCA1)  
- RefSeq: `NM_007294`  
- HGNC symbol: `BRCA1`  
Tools disagree on which format to use. Joining on gene ID is a common source of
silent data loss.

---
## Step 4 — Mathematical Explanation

**Median absolute deviation (MAD) for outlier detection:**

$$\text{MAD}(x) = \text{median}(|x_i - \text{median}(x)|)$$

A sample is flagged as outlier if its total counts deviate by more than $k \cdot \text{MAD}$
from the median (commonly $k = 3$). MAD is more robust than standard deviation because
it is not inflated by the outliers it's meant to detect.

---
## Step 5 — Computational Explanation

**Cleaning pipeline (in this order — order matters):**

1. Remove duplicate gene IDs (keep highest-variance row or sum counts)
2. Filter low-count genes
3. Flag/remove outlier samples
4. Log-transform (after adding pseudocount)
5. Normalize (CPM, TPM, or z-score)

Each step is a function with a clear input and output DataFrame. Log every step.

---
## Step 6 — Python Implementation

In [ ]:
import pandas as pd
import numpy as np
rng = np.random.default_rng(42)

In [ ]:
# Cell 6.1 — Build a dirty synthetic dataset
n_genes, n_samples = 200, 12
genes = [f"GENE_{i:04d}" for i in range(n_genes)]
samples = [f"S{i+1:02d}" for i in range(n_samples)]

counts = rng.negative_binomial(n=5, p=0.3, size=(n_genes, n_samples)).astype(float)

# Introduce problems:
# 1. Duplicate gene IDs
genes[10] = genes[5]   # gene 10 is a duplicate of gene 5
# 2. Low-count genes
counts[20:40, :] = rng.integers(0, 2, size=(20, n_samples))
# 3. One outlier sample (library size 10× lower)
counts[:, 7] = counts[:, 7] / 10
# 4. Some NaN values
counts[50, 2] = np.nan
counts[51, 3] = np.nan

df_raw = pd.DataFrame(counts, index=genes, columns=samples)
df_raw.index.name = "gene_id"

print(f"Raw shape: {df_raw.shape}")
print(f"NaN count: {df_raw.isna().sum().sum()}")
print(f"Duplicate gene IDs: {df_raw.index.duplicated().sum()}")
print(f"Library sizes (total counts per sample):")
print(df_raw.sum(axis=0).round(0).to_dict())

In [ ]:
# Cell 6.2 — Step 1: Resolve duplicate gene IDs
def resolve_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    For duplicate gene IDs, keep the row with higher total count.
    Logs how many duplicates were found and resolved.
    """
    n_dupes = df.index.duplicated().sum()
    if n_dupes == 0:
        print("No duplicate gene IDs found.")
        return df
    print(f"Found {n_dupes} duplicate gene IDs — keeping row with highest total count.")
    # Sum NaN-filled counts for ranking
    totals = df.fillna(0).sum(axis=1)
    df = df.copy()
    df["_total"] = totals
    df = df.sort_values("_total", ascending=False)
    df = df[~df.index.duplicated(keep="first")].drop(columns="_total")
    print(f"  Resolved → {df.shape[0]} unique genes")
    return df

df1 = resolve_duplicates(df_raw)
print(f"After dedup: {df1.shape}")

In [ ]:
# Cell 6.3 — Step 2: Filter low-count genes
def filter_low_counts(
    df: pd.DataFrame,
    min_count: float = 10,
    min_samples: int = 3,
) -> pd.DataFrame:
    """
    Remove genes where fewer than min_samples have count >= min_count.
    """
    keep_mask = (df >= min_count).sum(axis=1) >= min_samples
    n_removed = (~keep_mask).sum()
    print(f"Low-count filter: removed {n_removed} genes ({keep_mask.sum()} kept)")
    return df[keep_mask]

df2 = filter_low_counts(df1, min_count=10, min_samples=3)

In [ ]:
# Cell 6.4 — Step 3: Detect outlier samples using MAD
def flag_outlier_samples(
    df: pd.DataFrame,
    mad_threshold: float = 3.0,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Flag samples whose total library size deviates by > mad_threshold * MAD
    from the median library size.

    Returns
    -------
    df_clean : DataFrame without flagged samples
    flagged  : list of flagged sample names
    """
    lib_sizes = df.sum(axis=0)
    median_lib = lib_sizes.median()
    mad = (lib_sizes - median_lib).abs().median()
    z_mad = (lib_sizes - median_lib).abs() / (mad + 1e-9)
    flagged = list(z_mad[z_mad > mad_threshold].index)
    print(f"Outlier samples (MAD threshold={mad_threshold}): {flagged}")
    return df.drop(columns=flagged), flagged

df3, flagged = flag_outlier_samples(df2)
print(f"Shape after removing outliers: {df3.shape}")

In [ ]:
# Cell 6.5 — Step 4: Log2 + pseudocount and CPM
def log2_cpm(df: pd.DataFrame, pseudocount: float = 1.0) -> pd.DataFrame:
    """CPM normalize then log2-transform."""
    cpm = df.div(df.sum(axis=0)) * 1e6
    return np.log2(cpm + pseudocount)

df_clean = log2_cpm(df3.fillna(0))  # fill remaining NaN before transform
print(f"Clean dataset shape: {df_clean.shape}")
print(f"Value range: [{df_clean.min().min():.2f}, {df_clean.max().max():.2f}]")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Library size before and after outlier removal
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title in zip(axes,
    [df_raw.sum(axis=0), df3.sum(axis=0)],
    ["Raw library sizes", "After outlier removal"]):
    ax.bar(range(len(data)), data.values, color="steelblue", alpha=0.8)
    ax.axhline(data.median(), color="tomato", linestyle="--", label=f"Median: {data.median():.0f}")
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=45, ha="right")
    ax.set_ylabel("Total counts")
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Assemble `resolve_duplicates`, `filter_low_counts`, `flag_outlier_samples`, and
   `log2_cpm` into a single `clean_expression_matrix(df, **kwargs)` pipeline function
   in `compbio_utils/io.py`. It should return `(df_clean, report_dict)` where the
   report logs how many genes/samples were removed at each step.
2. Add `log2_cpm` to `compbio_utils/stats.py` with 3 unit tests.
3. What is the right way to handle NaN values in counts data — fill with 0, drop the
   gene, or impute? Write a one-paragraph justification.
4. Run the cleaning pipeline on the synthetic data and save `df_clean` to
   `datasets/processed/synthetic_clean.csv`.

---
## Quiz — Active Recall

1. Why is MAD preferred over standard deviation for outlier detection in library sizes?
2. What is the consequence of NOT filtering low-count genes before differential expression?
3. Why do we add a pseudocount before log-transforming?
4. What is the risk of silently dropping genes during a gene ID format mismatch?
5. In Cell 6.2, why do we sort by total count before deduplication rather than
   simply keeping the first occurrence?

---
## Reflection

**Date completed:** ____________________

> *[Is the full pipeline in compbio_utils/io.py with tests? Could you explain why each step goes in the order it does?]*

---
*Next: `10_matplotlib_seaborn_plotting.ipynb`*